<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/04_window_5min_episodes_FULL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB04_FULL — Detecção de Episódios Críticos por Célula

## 1. Contexto

Este notebook adapta o `NB04 — Detecção de Episódios Críticos` para o ramo `_FULL` do pipeline PPCOMP/DM.

A entrada oficial `_FULL` já é o Parquet model-facing re-selado pelo NB99_C, materializado em janelas de 5 minutos, com `raw_7` completo e nomes canônicos:

- `fail_rate`
- `n_events`
- `n_failed`
- `n_machines`
- `n_collections`
- `event_FAIL_count`
- `event_LOST_count`

A unidade experimental passa a ser a célula Borg (`cell_id ∈ {a,b,c,d,e,f,g,h}`). Cada célula é tratada como uma série temporal independente. As oito células **não devem ser concatenadas como uma única série temporal**.

## 2. Objetivo

Para cada célula ativa, este notebook:

1. lê o Parquet model-facing `_FULL`;
2. valida schema, nomes canônicos, ausência de colunas proibidas e integridade temporal;
3. recalcula o limiar oficial `TRAIN_M2S` por célula, usando apenas o trecho inicial de treino;
4. identifica janelas críticas por `fail_rate >= μ_train + 2σ_train`;
5. consolida janelas críticas contíguas em episódios críticos;
6. gera artefatos por célula e agregados;
7. emite um gate preliminar de viabilidade por célula para orientar os próximos notebooks.

## 3. Diferenças em relação ao NB04 canônico

| Aspecto | NB04 canônico | NB04_FULL |
|---|---|---|
| Entrada | `window_5min_series.parquet` canônico | Parquet model-facing `_FULL` do NB99_C |
| Unidade temporal | série única | uma série independente por `cell_id` |
| Limiar oficial | no NB04 original havia cenário global retrospectivo e sensibilidade train-only | no `_FULL`, o oficial é `TRAIN_M2S` por célula |
| Diagnóstico global | parte do cenário oficial/sensibilidade original | mantido apenas como diagnóstico por célula |
| Saídas | `02-datasets/03-features` e `04-reports` canônicos | `04-reports/99_FULL_downstream/04_FULL_episodes` |
| Escrita canônica | permitida no pipeline original | proibida no `_FULL` |
| Gate de modelabilidade | não aplicável | preliminar por célula, finalizado após rotulagem no NB06_FULL |

## 4. Política metodológica

- Não são introduzidas novas variáveis explicativas.
- A definição de criticidade permanece `μ + 2σ`.
- O limiar oficial do `_FULL` é calculado por célula e apenas no trecho de treino (`TRAIN_M2S_BY_CELL`).
- O cenário global por célula é preservado como diagnóstico retrospectivo, não como evidência oficial para modelagem.
- O NB04_FULL não faz engenharia de atributos, rotulagem supervisionada ou modelagem.

## 5. Como executar

Primeiro execute em piloto:

```python
ACTIVE_CELLS = ["a"]
```

Depois de validar os artefatos da célula `a`, execute completo:

```python
ACTIVE_CELLS = FULL_CELLS
```



In [2]:
# ============================================================
# NB04_FULL — Detecção de Episódios Críticos por Célula
# Pipeline PPCOMP_DM · ramo _FULL
#
# Escopo:
# - Ler o Parquet model-facing _FULL re-selado pelo NB99_C
# - Validar schema canônico e raw_7 completo
# - Processar cada cell_id como série temporal independente
# - Calcular limiar oficial TRAIN_M2S por célula
# - Consolidar janelas críticas contíguas em episódios críticos
# - Gerar artefatos por cell_<id> e aggregate
# - Gerar delta_vs_canonical.csv para rastreabilidade
#
# Importante:
# - Não concatena as oito células como uma única série temporal
# - Não escreve no caminho canônico 02-datasets/03-features
# - Não remapeia nomes: o NB99_C já entrega schema canônico
# - O cenário global por célula é diagnóstico, não oficial
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import subprocess
import importlib
import random
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────
# BLOCO 0 — Bootstrap Colab / Drive / Git
# ─────────────────────────────────────────────────────────────

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

DRIVE_ROOT = Path("/content/drive/MyDrive/Mestrado")
REPO_DIR = DRIVE_ROOT / "PPCOMP_DM"
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull)...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

importlib.invalidate_caches()

def log(msg: str) -> None:
    print(f"[NB04_FULL_detect_episodes] {msg}")

# ─────────────────────────────────────────────────────────────
# BLOCO 1 — Parâmetros _FULL
# ─────────────────────────────────────────────────────────────

WINDOW_MINUTES = 5
TRAIN_FRACTION_FOR_THRESHOLD = 0.80
THRESHOLD_POLICY = "TRAIN_M2S_BY_CELL"

FULL_CELLS = list("abcdefgh")

# Execução recomendada:
# 1) Piloto: ACTIVE_CELLS = ["a"]
# 2) Completa, após validação: ACTIVE_CELLS = FULL_CELLS
## ACTIVE_CELLS = ["a"]
ACTIVE_CELLS = FULL_CELLS

FULL_MODEL_FACING_PARQUET = (
    DRIVE_ROOT
    / "02-datasets"
    / "99-full"
    / "03-model-facing"
    / "window_5min_series_allcells_model_facing_000000000000.parquet"
)

FULL_REPORTS_ROOT = DRIVE_ROOT / "04-reports" / "99_FULL_downstream"
STAGE_DIR = FULL_REPORTS_ROOT / "04_FULL_episodes"
AGGREGATE_DIR = STAGE_DIR / "aggregate"

CANONICAL_FEATURES_DIR = DRIVE_ROOT / "02-datasets" / "03-features"

CANONICAL_TEMPORAL_CORE = [
    "fail_rate",
    "n_events",
    "n_failed",
    "n_machines",
    "n_collections",
    "event_FAIL_count",
    "event_LOST_count",
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_1h",
    "rolling_std_1h",
    "pct_change",
    "zscore_expanding",
]

RAW_7 = [
    "fail_rate",
    "n_events",
    "n_failed",
    "n_machines",
    "n_collections",
    "event_FAIL_count",
    "event_LOST_count",
]

REQUIRED_INPUT_COLUMNS = [
    "scenario_label",
    "cell_id",
    "bucket_id",
    "bucket_start_us",
    "bucket_start_ts",
    "n_events",
    "n_events_lifecycle",
    "n_failed",
    "n_update_pending_running",
    "n_machines",
    "n_collections",
    "event_FAIL_count",
    "event_LOST_count",
    "fail_rate",
    "fail_rate_lifecycle",
    "update_pending_running_share",
    "is_empty_bucket",
    "materialized_at_utc",
]

FORBIDDEN_INPUT_COLUMNS = {
    "n_events_all",
    "fail_rate_all_events",
    "type_5_FAIL_count",
    "type_8_LOST_count",
    "diagnostic_full_mu_2sigma_threshold_all_events",
    "diagnostic_full_mu_2sigma_threshold_lifecycle",
}

STAGE_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATE_DIR.mkdir(parents=True, exist_ok=True)

assert len(CANONICAL_TEMPORAL_CORE) == 14
assert set(RAW_7).issubset(set(CANONICAL_TEMPORAL_CORE))
assert FULL_MODEL_FACING_PARQUET.exists(), f"Parquet _FULL não encontrado: {FULL_MODEL_FACING_PARQUET}"
assert "99-full" in str(FULL_MODEL_FACING_PARQUET), "Entrada _FULL deve vir de 02-datasets/99-full"
assert "03-features" not in str(FULL_MODEL_FACING_PARQUET), "Entrada _FULL não pode vir do canônico 03-features"
assert "03-features" not in str(STAGE_DIR), "Saída _FULL não pode escrever em 03-features"
assert STAGE_DIR != CANONICAL_FEATURES_DIR

print("DRIVE_ROOT =", DRIVE_ROOT)
print("FULL_MODEL_FACING_PARQUET =", FULL_MODEL_FACING_PARQUET)
print("STAGE_DIR =", STAGE_DIR)
print("AGGREGATE_DIR =", AGGREGATE_DIR)
print("ACTIVE_CELLS =", ACTIVE_CELLS)

# ─────────────────────────────────────────────────────────────
# BLOCO 2 — Funções auxiliares
# ─────────────────────────────────────────────────────────────

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def compute_threshold(values: pd.Series) -> dict:
    """Calcula média, desvio-padrão populacional e limiar μ + 2σ."""
    values = pd.Series(values).astype(float)
    mu = float(values.mean())
    sigma = float(values.std(ddof=0))
    threshold = float(mu + 2 * sigma)

    return {
        "mu": mu,
        "sigma": sigma,
        "threshold": threshold,
    }


def detect_episodes_from_flag(
    df: pd.DataFrame,
    critical_col: str,
    threshold_info: dict,
    scenario_name: str,
    cell_id: str,
) -> pd.DataFrame:
    """
    Consolida janelas críticas contíguas em episódios.

    Cada episódio é uma sequência máxima de linhas consecutivas com critical_col == 1.
    """
    episodes = []
    in_episode = False
    start_idx = None

    flags = df[critical_col].to_numpy()

    for i, flag in enumerate(flags):
        if flag == 1 and not in_episode:
            in_episode = True
            start_idx = i
        elif flag == 0 and in_episode:
            episodes.append((start_idx, i - 1))
            in_episode = False

    if in_episode:
        episodes.append((start_idx, len(df) - 1))

    rows = []

    for ep_id, (s, e) in enumerate(episodes, start=1):
        seg = df.iloc[s:e + 1]

        rows.append({
            "scenario": scenario_name,
            "cell_id": cell_id,
            "episode_id": ep_id,
            "start_bucket": int(seg["bucket_id"].iloc[0]),
            "end_bucket": int(seg["bucket_id"].iloc[-1]),
            "start_bucket_start_us": int(seg["bucket_start_us"].iloc[0]),
            "end_bucket_start_us": int(seg["bucket_start_us"].iloc[-1]),
            "duration_windows": int(len(seg)),
            "duration_minutes": int(len(seg) * WINDOW_MINUTES),
            "max_fail_rate": float(seg["fail_rate"].max()),
            "mean_fail_rate": float(seg["fail_rate"].mean()),
            "max_failed": int(seg["n_failed"].max()),
            "mean_failed": float(seg["n_failed"].mean()),
            "sum_failed": int(seg["n_failed"].sum()),
            "max_events": int(seg["n_events"].max()),
            "mean_events": float(seg["n_events"].mean()),
            "threshold": float(threshold_info["threshold"]),
            "mu": float(threshold_info["mu"]),
            "sigma": float(threshold_info["sigma"]),
        })

    return pd.DataFrame(rows)


def summarize_episode_result(
    df_scored: pd.DataFrame,
    episodes_df: pd.DataFrame,
    critical_col: str,
    threshold_info: dict,
    scenario_name: str,
    cell_id: str,
    train_cutoff_idx: int | None = None,
) -> dict:
    """Resume a classificação crítica e a distribuição de duração dos episódios."""
    critical_windows = int(df_scored[critical_col].sum())
    episodes_detected = int(len(episodes_df))

    if episodes_detected > 0:
        one_window_episodes = int((episodes_df["duration_windows"] == 1).sum())
        duration_windows_mean = float(episodes_df["duration_windows"].mean())
        duration_windows_median = float(episodes_df["duration_windows"].median())
        duration_windows_max = int(episodes_df["duration_windows"].max())
        duration_minutes_mean = float(episodes_df["duration_minutes"].mean())
        duration_minutes_median = float(episodes_df["duration_minutes"].median())
        duration_minutes_max = int(episodes_df["duration_minutes"].max())
    else:
        one_window_episodes = 0
        duration_windows_mean = 0.0
        duration_windows_median = 0.0
        duration_windows_max = 0
        duration_minutes_mean = 0.0
        duration_minutes_median = 0.0
        duration_minutes_max = 0

    out = {
        "scenario": scenario_name,
        "cell_id": cell_id,
        "mu": float(threshold_info["mu"]),
        "sigma": float(threshold_info["sigma"]),
        "threshold": float(threshold_info["threshold"]),
        "critical_windows": critical_windows,
        "episodes_detected": episodes_detected,
        "one_window_episodes": one_window_episodes,
        "duration_windows_mean": duration_windows_mean,
        "duration_windows_median": duration_windows_median,
        "duration_windows_max": duration_windows_max,
        "duration_minutes_mean": duration_minutes_mean,
        "duration_minutes_median": duration_minutes_median,
        "duration_minutes_max": duration_minutes_max,
    }

    if train_cutoff_idx is not None:
        train_flags = df_scored[critical_col].iloc[:train_cutoff_idx]
        test_flags = df_scored[critical_col].iloc[train_cutoff_idx:]
        out.update({
            "train_cutoff_idx": int(train_cutoff_idx),
            "critical_windows_train_segment": int(train_flags.sum()),
            "critical_windows_test_segment": int(test_flags.sum()),
            "train_rows": int(len(train_flags)),
            "test_rows": int(len(test_flags)),
        })

    return out


def make_cell_dir(cell_id: str) -> Path:
    cell_dir = STAGE_DIR / f"cell_{cell_id}"
    cell_dir.mkdir(parents=True, exist_ok=True)
    return cell_dir


def write_stage_manifest(stage_dir: Path) -> Path:
    manifest_file = stage_dir / "aggregate" / "04_FULL_artifact_manifest_sha256.csv"
    rows = []
    for p in sorted(stage_dir.rglob("*")):
        if p.is_file() and p != manifest_file:
            rows.append({
                "relative_path": str(p.relative_to(stage_dir)).replace("\\", "/"),
                "size_bytes": int(p.stat().st_size),
                "sha256": sha256_file(p),
                "last_modified_utc": datetime.fromtimestamp(
                    p.stat().st_mtime, tz=timezone.utc
                ).isoformat(),
            })
    pd.DataFrame(rows).to_csv(manifest_file, index=False)
    return manifest_file


def write_delta_vs_canonical() -> Path:
    rows = [
        {
            "notebook_full": "04_window_5min_episodes_FULL.ipynb",
            "notebook_canonical": "04_window_5min_episodes.ipynb",
            "change_type": "FULL_DATASET_INPUT",
            "description": "Entrada alterada para o Parquet model-facing _FULL re-selado pelo NB99_C.",
            "justified": True,
        },
        {
            "notebook_full": "04_window_5min_episodes_FULL.ipynb",
            "notebook_canonical": "04_window_5min_episodes.ipynb",
            "change_type": "CELL_LOOP",
            "description": "Execução parametrizada por cell_id, sem concatenar as células como uma única série temporal.",
            "justified": True,
        },
        {
            "notebook_full": "04_window_5min_episodes_FULL.ipynb",
            "notebook_canonical": "04_window_5min_episodes.ipynb",
            "change_type": "TRAIN_ONLY_THRESHOLD_BY_CELL",
            "description": "Limiar oficial TRAIN_M2S calculado por célula e apenas no trecho de treino.",
            "justified": True,
        },
        {
            "notebook_full": "04_window_5min_episodes_FULL.ipynb",
            "notebook_canonical": "04_window_5min_episodes.ipynb",
            "change_type": "OUTPUT_PREFIX",
            "description": "Artefatos escritos em 04-reports/99_FULL_downstream/04_FULL_episodes, com prefixo 04_FULL.",
            "justified": True,
        },
        {
            "notebook_full": "04_window_5min_episodes_FULL.ipynb",
            "notebook_canonical": "04_window_5min_episodes.ipynb",
            "change_type": "AGGREGATION_BY_CELL",
            "description": "Geração de summaries por célula e consolidação em aggregate.",
            "justified": True,
        },
        {
            "notebook_full": "04_window_5min_episodes_FULL.ipynb",
            "notebook_canonical": "04_window_5min_episodes.ipynb",
            "change_type": "REPORTING_ONLY",
            "description": "Gate preliminar de suporte episódico por célula; modelabilidade final depende do NB06_FULL.",
            "justified": True,
        },
    ]
    out = AGGREGATE_DIR / "04_FULL_delta_vs_canonical.csv"
    pd.DataFrame(rows).to_csv(out, index=False)
    return out


# ─────────────────────────────────────────────────────────────
# BLOCO 3 — Leitura e validação da entrada _FULL
# ─────────────────────────────────────────────────────────────

df_all = pd.read_parquet(FULL_MODEL_FACING_PARQUET)

missing_required = [c for c in REQUIRED_INPUT_COLUMNS if c not in df_all.columns]
missing_raw7 = [c for c in RAW_7 if c not in df_all.columns]
forbidden_found = [c for c in FORBIDDEN_INPUT_COLUMNS if c in df_all.columns]
type_prefix_found = [c for c in df_all.columns if c.startswith("type_")]
diagnostic_found = [c for c in df_all.columns if c.startswith("diagnostic_full_mu_2sigma")]

assert not missing_required, f"Colunas required ausentes no Parquet _FULL: {missing_required}"
assert not missing_raw7, f"raw_7 incompleto no Parquet _FULL: {missing_raw7}"
assert not forbidden_found, f"Colunas proibidas encontradas no Parquet _FULL: {forbidden_found}"
assert not type_prefix_found, f"Colunas type_* encontradas no Parquet _FULL: {type_prefix_found}"
assert not diagnostic_found, f"Colunas diagnostic_full_mu_2sigma* encontradas no Parquet _FULL: {diagnostic_found}"

assert sorted(df_all["cell_id"].unique().tolist()) == FULL_CELLS
assert set(ACTIVE_CELLS).issubset(set(FULL_CELLS))

dup_count = int(df_all.duplicated(["cell_id", "bucket_id"]).sum())
assert dup_count == 0, f"Duplicidade em (cell_id, bucket_id): {dup_count}"

# Validação de fail_rate entregue pelo NB99_C.
fail_rate_recomputed = np.where(
    df_all["n_events"].to_numpy() > 0,
    df_all["n_failed"].to_numpy() / df_all["n_events"].to_numpy(),
    0.0,
)
fail_rate_max_abs_diff = float(
    np.nanmax(np.abs(df_all["fail_rate"].to_numpy(dtype=float) - fail_rate_recomputed))
)
assert np.isclose(fail_rate_max_abs_diff, 0.0, atol=1e-12), (
    f"fail_rate difere do recomputado: {fail_rate_max_abs_diff}"
)

log(f"Entrada _FULL carregada: shape={df_all.shape}")
log(f"Células disponíveis: {sorted(df_all['cell_id'].unique().tolist())}")
log(f"fail_rate max abs diff vs recomputado: {fail_rate_max_abs_diff}")

# ─────────────────────────────────────────────────────────────
# BLOCO 4 — Processamento por célula
# ─────────────────────────────────────────────────────────────

def process_cell(cell_id: str) -> dict:
    log(f"Iniciando célula {cell_id}")

    cell_dir = make_cell_dir(cell_id)

    df_cell = (
        df_all[df_all["cell_id"] == cell_id]
        .sort_values("bucket_id")
        .reset_index(drop=True)
        .copy()
    )

    assert len(df_cell) > 0, f"Célula sem linhas: {cell_id}"
    assert df_cell["bucket_id"].is_monotonic_increasing
    assert df_cell["bucket_id"].is_unique

    bucket_diff = df_cell["bucket_id"].diff().dropna()
    n_internal_gaps_gt1 = int((bucket_diff > 1).sum())
    assert n_internal_gaps_gt1 == 0, (
        f"A série da célula {cell_id} possui lacunas internas > 1."
    )

    # Recalcula a fail_rate apenas para validar e estabilizar tipo.
    den = df_cell["n_events"].replace(0, np.nan)
    fail_rate_calc = (df_cell["n_failed"] / den).fillna(0.0).astype(float)
    max_abs_diff = float(np.nanmax(np.abs(df_cell["fail_rate"].astype(float) - fail_rate_calc)))
    assert np.isclose(max_abs_diff, 0.0, atol=1e-12), (
        f"fail_rate inválida na célula {cell_id}: max_abs_diff={max_abs_diff}"
    )
    df_cell["fail_rate"] = fail_rate_calc

    series_rows = int(len(df_cell))
    bucket_min = int(df_cell["bucket_id"].min())
    bucket_max = int(df_cell["bucket_id"].max())
    zero_event_windows = int((df_cell["n_events"] == 0).sum())
    nonzero_event_windows = int((df_cell["n_events"] > 0).sum())

    train_cutoff_idx = int(np.floor(series_rows * TRAIN_FRACTION_FOR_THRESHOLD))
    train_cutoff_idx = max(1, min(train_cutoff_idx, series_rows))

    df_train_portion = df_cell.iloc[:train_cutoff_idx].copy()

    train_threshold_info = compute_threshold(df_train_portion["fail_rate"])
    global_threshold_info = compute_threshold(df_cell["fail_rate"])

    df_scored = df_cell.copy()
    df_scored["threshold_policy"] = THRESHOLD_POLICY
    df_scored["threshold_train_mu"] = float(train_threshold_info["mu"])
    df_scored["threshold_train_sigma"] = float(train_threshold_info["sigma"])
    df_scored["threshold_train_m2s"] = float(train_threshold_info["threshold"])
    df_scored["train_cutoff_idx"] = int(train_cutoff_idx)
    df_scored["train_cutoff_bucket_id"] = int(df_cell["bucket_id"].iloc[train_cutoff_idx - 1])
    df_scored["is_critical"] = (
        df_scored["fail_rate"] >= train_threshold_info["threshold"]
    ).astype("int8")

    # Diagnóstico retrospectivo por célula. Não usar como critério oficial.
    df_scored["diagnostic_global_mu"] = float(global_threshold_info["mu"])
    df_scored["diagnostic_global_sigma"] = float(global_threshold_info["sigma"])
    df_scored["diagnostic_global_m2s_threshold"] = float(global_threshold_info["threshold"])
    df_scored["is_critical_global_diagnostic"] = (
        df_scored["fail_rate"] >= global_threshold_info["threshold"]
    ).astype("int8")

    episodes_train_m2s = detect_episodes_from_flag(
        df=df_scored,
        critical_col="is_critical",
        threshold_info=train_threshold_info,
        scenario_name="train_m2s_by_cell",
        cell_id=cell_id,
    )

    episodes_global_diagnostic = detect_episodes_from_flag(
        df=df_scored,
        critical_col="is_critical_global_diagnostic",
        threshold_info=global_threshold_info,
        scenario_name="global_full_series_by_cell_diagnostic",
        cell_id=cell_id,
    )

    summary_train_m2s = summarize_episode_result(
        df_scored=df_scored,
        episodes_df=episodes_train_m2s,
        critical_col="is_critical",
        threshold_info=train_threshold_info,
        scenario_name="train_m2s_by_cell",
        cell_id=cell_id,
        train_cutoff_idx=train_cutoff_idx,
    )

    summary_global_diag = summarize_episode_result(
        df_scored=df_scored,
        episodes_df=episodes_global_diagnostic,
        critical_col="is_critical_global_diagnostic",
        threshold_info=global_threshold_info,
        scenario_name="global_full_series_by_cell_diagnostic",
        cell_id=cell_id,
        train_cutoff_idx=train_cutoff_idx,
    )

    threshold_sensitivity = pd.DataFrame([
        summary_train_m2s,
        summary_global_diag,
    ])
    threshold_sensitivity["delta_critical_windows_vs_train_m2s"] = (
        threshold_sensitivity["critical_windows"]
        - summary_train_m2s["critical_windows"]
    )
    threshold_sensitivity["delta_episodes_vs_train_m2s"] = (
        threshold_sensitivity["episodes_detected"]
        - summary_train_m2s["episodes_detected"]
    )
    threshold_sensitivity["delta_threshold_vs_train_m2s"] = (
        threshold_sensitivity["threshold"]
        - summary_train_m2s["threshold"]
    )

    # Gate preliminar. A modelabilidade supervisionada final depende do NB06_FULL.
    if summary_train_m2s["episodes_detected"] == 0:
        preliminary_modeling_status = "DESCRIPTIVE_ONLY_NO_EPISODES_IN_NB04"
        status_reason = "Nenhum episódio crítico detectado no NB04_FULL sob TRAIN_M2S por célula."
    else:
        preliminary_modeling_status = "HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS"
        status_reason = (
            "Há episódios críticos detectados. A modelabilidade supervisionada "
            "será decidida após a rotulagem do NB06_FULL."
        )

    scored_file = cell_dir / f"04_FULL_window_5min_series_scored_TRAIN_M2S_cell_{cell_id}.parquet"
    episodes_file = cell_dir / f"04_FULL_episodes_TRAIN_M2S_cell_{cell_id}.parquet"
    episodes_global_file = cell_dir / f"04_FULL_episodes_global_diagnostic_cell_{cell_id}.parquet"
    sensitivity_file = cell_dir / f"04_FULL_threshold_sensitivity_cell_{cell_id}.csv"
    summary_file = cell_dir / f"04_FULL_detect_episodes_summary_cell_{cell_id}.json"

    df_scored.to_parquet(scored_file, compression="snappy", index=False)
    episodes_train_m2s.to_parquet(episodes_file, compression="snappy", index=False)
    episodes_global_diagnostic.to_parquet(episodes_global_file, compression="snappy", index=False)
    threshold_sensitivity.to_csv(sensitivity_file, index=False)

    cell_summary = {
        "notebook": "NB04_FULL",
        "purpose": "detect_critical_episodes_by_cell",
        "cell_id": cell_id,
        "threshold_policy_official": THRESHOLD_POLICY,
        "metric": "fail_rate",
        "input_series_file": str(FULL_MODEL_FACING_PARQUET),
        "series_rows": series_rows,
        "bucket_id_min": bucket_min,
        "bucket_id_max": bucket_max,
        "zero_event_windows": zero_event_windows,
        "nonzero_event_windows": nonzero_event_windows,
        "train_fraction_for_threshold": float(TRAIN_FRACTION_FOR_THRESHOLD),
        "train_cutoff_idx": int(train_cutoff_idx),
        "train_cutoff_bucket_id": int(df_cell["bucket_id"].iloc[train_cutoff_idx - 1]),
        "official_result": summary_train_m2s,
        "global_diagnostic_result": summary_global_diag,
        "preliminary_modeling_status": preliminary_modeling_status,
        "status_reason": status_reason,
        "output_series_scored_file": str(scored_file),
        "output_episodes_file": str(episodes_file),
        "output_episodes_global_diagnostic_file": str(episodes_global_file),
        "output_threshold_sensitivity_file": str(sensitivity_file),
    }

    write_json(summary_file, cell_summary)

    log(
        f"Célula {cell_id}: rows={series_rows}, "
        f"critical_windows={summary_train_m2s['critical_windows']}, "
        f"episodes={summary_train_m2s['episodes_detected']}, "
        f"threshold={summary_train_m2s['threshold']:.8f}, "
        f"status={preliminary_modeling_status}"
    )

    return cell_summary


cell_summaries = [process_cell(cell_id) for cell_id in ACTIVE_CELLS]

# ─────────────────────────────────────────────────────────────
# BLOCO 5 — Consolidação aggregate
# ─────────────────────────────────────────────────────────────

summary_rows = []
threshold_rows = []
gate_rows = []
scored_frames = []
episode_frames = []

for s in cell_summaries:
    cell_id = s["cell_id"]
    cell_dir = make_cell_dir(cell_id)

    summary_rows.append({
        "cell_id": cell_id,
        "series_rows": s["series_rows"],
        "bucket_id_min": s["bucket_id_min"],
        "bucket_id_max": s["bucket_id_max"],
        "zero_event_windows": s["zero_event_windows"],
        "nonzero_event_windows": s["nonzero_event_windows"],
        "train_cutoff_idx": s["train_cutoff_idx"],
        "train_cutoff_bucket_id": s["train_cutoff_bucket_id"],
        "threshold_policy": s["threshold_policy_official"],
        "train_mu": s["official_result"]["mu"],
        "train_sigma": s["official_result"]["sigma"],
        "threshold_train_m2s": s["official_result"]["threshold"],
        "critical_windows": s["official_result"]["critical_windows"],
        "episodes_detected": s["official_result"]["episodes_detected"],
        "one_window_episodes": s["official_result"]["one_window_episodes"],
        "duration_windows_mean": s["official_result"]["duration_windows_mean"],
        "duration_windows_median": s["official_result"]["duration_windows_median"],
        "duration_windows_max": s["official_result"]["duration_windows_max"],
        "critical_windows_train_segment": s["official_result"]["critical_windows_train_segment"],
        "critical_windows_test_segment": s["official_result"]["critical_windows_test_segment"],
        "diagnostic_global_threshold": s["global_diagnostic_result"]["threshold"],
        "diagnostic_global_critical_windows": s["global_diagnostic_result"]["critical_windows"],
        "diagnostic_global_episodes": s["global_diagnostic_result"]["episodes_detected"],
    })

    threshold_rows.append({
        "cell_id": cell_id,
        "threshold_policy": s["threshold_policy_official"],
        "train_fraction": float(TRAIN_FRACTION_FOR_THRESHOLD),
        "train_cutoff_idx": s["train_cutoff_idx"],
        "train_cutoff_bucket_id": s["train_cutoff_bucket_id"],
        "train_mu": s["official_result"]["mu"],
        "train_sigma": s["official_result"]["sigma"],
        "threshold_train_m2s": s["official_result"]["threshold"],
        "global_mu_diagnostic": s["global_diagnostic_result"]["mu"],
        "global_sigma_diagnostic": s["global_diagnostic_result"]["sigma"],
        "threshold_global_m2s_diagnostic": s["global_diagnostic_result"]["threshold"],
    })

    gate_rows.append({
        "cell_id": cell_id,
        "n_janelas_criticas": s["official_result"]["critical_windows"],
        "n_episodios": s["official_result"]["episodes_detected"],
        "n_positivos_train": None,
        "n_positivos_test": None,
        "status_modelagem": s["preliminary_modeling_status"],
        "justificativa": s["status_reason"],
    })

    scored_frames.append(pd.read_parquet(
        cell_dir / f"04_FULL_window_5min_series_scored_TRAIN_M2S_cell_{cell_id}.parquet"
    ))
    episode_frames.append(pd.read_parquet(
        cell_dir / f"04_FULL_episodes_TRAIN_M2S_cell_{cell_id}.parquet"
    ))

episode_summary_by_cell = pd.DataFrame(summary_rows).sort_values("cell_id")
thresholds_by_cell = pd.DataFrame(threshold_rows).sort_values("cell_id")
modelability_gate_by_cell = pd.DataFrame(gate_rows).sort_values("cell_id")

episodes_all_active = (
    pd.concat(episode_frames, ignore_index=True)
    if episode_frames
    else pd.DataFrame()
)
scored_all_active = (
    pd.concat(scored_frames, ignore_index=True)
    if scored_frames
    else pd.DataFrame()
)

episode_summary_by_cell_file = AGGREGATE_DIR / "04_FULL_episode_summary_by_cell.csv"
thresholds_by_cell_file = AGGREGATE_DIR / "04_FULL_thresholds_by_cell.csv"
modelability_gate_file = AGGREGATE_DIR / "04_FULL_modelability_gate_by_cell.csv"
episodes_all_file = AGGREGATE_DIR / "04_FULL_episodes_TRAIN_M2S_active_cells.parquet"
scored_all_file = AGGREGATE_DIR / "04_FULL_window_5min_series_scored_TRAIN_M2S_active_cells.parquet"
summary_file = AGGREGATE_DIR / "04_FULL_detect_episodes_summary.json"

episode_summary_by_cell.to_csv(episode_summary_by_cell_file, index=False)
thresholds_by_cell.to_csv(thresholds_by_cell_file, index=False)
modelability_gate_by_cell.to_csv(modelability_gate_file, index=False)
episodes_all_active.to_parquet(episodes_all_file, compression="snappy", index=False)
scored_all_active.to_parquet(scored_all_file, compression="snappy", index=False)

delta_file = write_delta_vs_canonical()

aggregate_summary = {
    "notebook": "NB04_FULL",
    "purpose": "detect_critical_episodes_by_cell",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_series_file": str(FULL_MODEL_FACING_PARQUET),
    "input_sha256": sha256_file(FULL_MODEL_FACING_PARQUET),
    "active_cells": ACTIVE_CELLS,
    "full_cells_available": FULL_CELLS,
    "threshold_policy_official": THRESHOLD_POLICY,
    "train_fraction_for_threshold": float(TRAIN_FRACTION_FOR_THRESHOLD),
    "window_minutes": int(WINDOW_MINUTES),
    "canonical_temporal_core": CANONICAL_TEMPORAL_CORE,
    "raw_7": RAW_7,
    "fail_rate_max_abs_diff_vs_recomputed": fail_rate_max_abs_diff,
    "cell_summaries": cell_summaries,
    "outputs": {
        "episode_summary_by_cell": str(episode_summary_by_cell_file),
        "thresholds_by_cell": str(thresholds_by_cell_file),
        "modelability_gate_by_cell": str(modelability_gate_file),
        "episodes_active_cells": str(episodes_all_file),
        "scored_series_active_cells": str(scored_all_file),
        "delta_vs_canonical": str(delta_file),
    },
    "notes": [
        "Cenário oficial _FULL: TRAIN_M2S calculado por célula sobre o segmento de treino.",
        "Cenário global por célula é diagnóstico retrospectivo, não evidência oficial de modelagem.",
        "n_positivos_train/test ficam pendentes até o NB06_FULL, que realiza a rotulagem supervisionada.",
    ],
}

write_json(summary_file, aggregate_summary)

manifest_file = write_stage_manifest(STAGE_DIR)

# ─────────────────────────────────────────────────────────────
# BLOCO 6 — Resumo final na tela
# ─────────────────────────────────────────────────────────────

print("\n=== RESUMO FINAL — NB04_FULL ===")
print("ACTIVE_CELLS:", ACTIVE_CELLS)
print("Entrada:", FULL_MODEL_FACING_PARQUET)
print("Saída:", STAGE_DIR)
print("Threshold policy oficial:", THRESHOLD_POLICY)
print()

print("[Resumo por célula]")
display(episode_summary_by_cell)

print("\n[Thresholds por célula]")
display(thresholds_by_cell)

print("\n[Gate preliminar de modelabilidade]")
display(modelability_gate_by_cell)

print("\n[Head episódios TRAIN_M2S — células ativas]")
display(episodes_all_active.head())

print("\nArtefatos aggregate:")
for p in [
    episode_summary_by_cell_file,
    thresholds_by_cell_file,
    modelability_gate_file,
    episodes_all_file,
    scored_all_file,
    delta_file,
    summary_file,
    manifest_file,
]:
    print("-", p)


[Bootstrap] Google Drive já montado.
[Bootstrap] Atualizando repositório (git pull)...
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
DRIVE_ROOT = /content/drive/MyDrive/Mestrado
FULL_MODEL_FACING_PARQUET = /content/drive/MyDrive/Mestrado/02-datasets/99-full/03-model-facing/window_5min_series_allcells_model_facing_000000000000.parquet
STAGE_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes
AGGREGATE_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate
ACTIVE_CELLS = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
[NB04_FULL_detect_episodes] Entrada _FULL carregada: shape=(71424, 28)
[NB04_FULL_detect_episodes] Células disponíveis: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
[NB04_FULL_detect_episodes] fail_rate max abs diff vs recomputado: 0.0
[NB04_FULL_detect_episodes] Iniciando célula a
[NB04_FULL_detect_episodes] Célula a: rows=8928, critical_windows=441, episodes=191, threshold=0.02904750, status=HAS_E

,cell_id,series_rows,bucket_id_min,bucket_id_max,zero_event_windows,nonzero_event_windows,train_cutoff_idx,train_cutoff_bucket_id,threshold_policy,train_mu,...,episodes_detected,one_window_episodes,duration_windows_mean,duration_windows_median,duration_windows_max,critical_windows_train_segment,critical_windows_test_segment,diagnostic_global_threshold,diagnostic_global_critical_windows,diagnostic_global_episodes
0,a,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.011411,...,191,107,2.308901,1.0,36,366,75,0.029038,441,191
1,b,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.007800,...,219,134,2.182648,1.0,25,372,106,0.032067,463,215
2,c,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.007989,...,178,95,2.134831,1.0,14,339,41,0.017984,420,201
3,d,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.009960,...,130,56,3.176923,2.0,22,410,3,0.034064,493,125
4,e,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.003529,...,110,73,2.227273,1.0,22,223,22,0.010563,251,110
5,f,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.004168,...,135,87,1.829630,1.0,8,195,52,0.012550,245,134
6,g,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.003657,...,295,210,2.040678,1.0,13,311,291,0.022442,320,140
7,h,8928,2,8929,0,8928,7142,7143,TRAIN_M2S_BY_CELL,0.003464,...,225,177,1.302222,1.0,11,248,45,0.013563,309,233



[Thresholds por célula]


,cell_id,threshold_policy,train_fraction,train_cutoff_idx,train_cutoff_bucket_id,train_mu,train_sigma,threshold_train_m2s,global_mu_diagnostic,global_sigma_diagnostic,threshold_global_m2s_diagnostic
0,a,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.011411,0.008818,0.029047,0.011200,0.008919,0.029038
1,b,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.007800,0.011853,0.031506,0.007725,0.012171,0.032067
2,c,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.007989,0.005288,0.018565,0.007714,0.005135,0.017984
3,d,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.009960,0.013788,0.037535,0.008699,0.012682,0.034064
4,e,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.003529,0.003616,0.010760,0.003540,0.003511,0.010563
5,f,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.004168,0.004156,0.012479,0.003990,0.004280,0.012550
6,g,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.003657,0.005595,0.014847,0.004639,0.008902,0.022442
7,h,TRAIN_M2S_BY_CELL,0.8,7142,7143,0.003464,0.005264,0.013992,0.003434,0.005064,0.013563



[Gate preliminar de modelabilidade]


,cell_id,n_janelas_criticas,n_episodios,n_positivos_train,n_positivos_test,status_modelagem,justificativa
0,a,441,191,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...
1,b,478,219,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...
2,c,380,178,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...
3,d,413,130,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...
4,e,245,110,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...
5,f,247,135,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...
6,g,602,295,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...
7,h,293,225,None,None,HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS,Há episódios críticos detectados. A modelabili...



[Head episódios TRAIN_M2S — células ativas]


,scenario,cell_id,episode_id,start_bucket,end_bucket,start_bucket_start_us,end_bucket_start_us,duration_windows,duration_minutes,max_fail_rate,mean_fail_rate,max_failed,mean_failed,sum_failed,max_events,mean_events,threshold,mu,sigma
0,train_m2s_by_cell,a,1,272,272,81600000000,81600000000,1,5,0.032857,0.032857,7367,7367.0,7367,224212,224212.0,0.029047,0.011411,0.008818
1,train_m2s_by_cell,a,2,492,492,147600000000,147600000000,1,5,0.037784,0.037784,5169,5169.0,5169,136805,136805.0,0.029047,0.011411,0.008818
2,train_m2s_by_cell,a,3,687,691,206100000000,207300000000,5,25,0.044254,0.036761,5222,4082.8,20414,122914,110123.8,0.029047,0.011411,0.008818
3,train_m2s_by_cell,a,4,694,694,208200000000,208200000000,1,5,0.045592,0.045592,5776,5776.0,5776,126690,126690.0,0.029047,0.011411,0.008818
4,train_m2s_by_cell,a,5,710,710,213000000000,213000000000,1,5,0.038339,0.038339,7094,7094.0,7094,185033,185033.0,0.029047,0.011411,0.008818



Artefatos aggregate:
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_episode_summary_by_cell.csv
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_thresholds_by_cell.csv
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_modelability_gate_by_cell.csv
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_episodes_TRAIN_M2S_active_cells.parquet
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_window_5min_series_scored_TRAIN_M2S_active_cells.parquet
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_delta_vs_canonical.csv
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_detect_episodes_summary.json
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_do

## Conclusão do NB04_FULL — Detecção de episódios críticos

O notebook **NB04_FULL** foi executado com sucesso sobre a base `_FULL`, considerando as oito células disponíveis do conjunto materializado: `a`, `b`, `c`, `d`, `e`, `f`, `g` e `h`.

A execução completa foi precedida por uma validação piloto da célula `a`, seguida da execução FULL para todas as células. A entrada utilizada foi o arquivo model-facing `_FULL` com **71.424 janelas de 5 minutos**, correspondentes a **8 células × 8.928 janelas por célula**. Todas as células apresentaram `bucket_id` no intervalo de **2 a 8.929**, sem janelas vazias e sem divergência no recálculo de `fail_rate`.

A política oficial de limiar adotada foi:

`TRAIN_M2S_BY_CELL`

Isto é, para cada célula, o limiar crítico foi calculado apenas com base no segmento de treino, usando a regra:

`threshold = μ_train + 2σ_train`

Essa escolha preserva a consistência temporal da etapa, evitando vazamento de informação do segmento de teste para a definição do limiar.

### Resumo por célula

| Célula | Janelas | Threshold oficial | Janelas críticas | Episódios | Episódios de 1 janela | Críticas treino | Críticas teste | Maior episódio |
| ------ | ------: | ----------------: | ---------------: | --------: | --------------------: | --------------: | -------------: | -------------: |
| a      |   8.928 |          0,029047 |              441 |       191 |                   107 |             366 |             75 |     36 janelas |
| b      |   8.928 |          0,031506 |              478 |       219 |                   134 |             372 |            106 |     25 janelas |
| c      |   8.928 |          0,018565 |              380 |       178 |                    95 |             339 |             41 |     14 janelas |
| d      |   8.928 |          0,037535 |              413 |       130 |                    56 |             410 |              3 |     22 janelas |
| e      |   8.928 |          0,010760 |              245 |       110 |                    73 |             223 |             22 |     22 janelas |
| f      |   8.928 |          0,012479 |              247 |       135 |                    87 |             195 |             52 |      8 janelas |
| g      |   8.928 |          0,014847 |              602 |       295 |                   210 |             311 |            291 |     13 janelas |
| h      |   8.928 |          0,013992 |              293 |       225 |                   177 |             248 |             45 |     11 janelas |

### Totais consolidados

No conjunto das oito células, foram detectadas:

* **71.424 janelas avaliadas**
* **3.099 janelas críticas**
* **1.483 episódios críticos**
* **939 episódios de uma única janela**
* **2.464 janelas críticas no segmento de treino**
* **635 janelas críticas no segmento de teste**

Todas as células receberam o status preliminar:

`HAS_EPISODE_SUPPORT_PENDING_NB06_LABELS`

Esse status indica que houve suporte empírico suficiente para identificação de episódios críticos no NB04_FULL. A decisão final sobre a viabilidade da modelagem supervisionada permanece corretamente postergada para as etapas seguintes de construção e validação dos rótulos supervisionados.

### Validações realizadas

Foram validados os seguintes pontos:

1. As oito células esperadas foram processadas.
2. Cada célula possui exatamente **8.928 janelas**.
3. O total consolidado fecha em **71.424 janelas**.
4. Não há janelas vazias.
5. O campo `fail_rate` foi recomputado sem divergência.
6. A coluna `fail_rate` foi preservada em tipo numérico de dupla precisão.
7. A coluna `fail_rate_lifecycle` permaneceu disponível para análises de sensibilidade.
8. A soma das durações dos episódios coincide com o número de janelas críticas por célula.
9. A quantidade de blocos contíguos críticos coincide com o número de episódios detectados.
10. Os artefatos por célula foram gerados para série pontuada, episódios oficiais, diagnóstico global e sensibilidade de limiar.
11. A política oficial `TRAIN_M2S_BY_CELL` foi preservada em todas as células.
12. Todas as células apresentaram suporte preliminar para a etapa seguinte.

### Interpretação metodológica

Os resultados confirmam que a transição do cenário canônico para o cenário `_FULL` preservou a lógica metodológica do NB04, agora aplicada célula a célula. A detecção produziu episódios críticos em todas as células, com heterogeneidade esperada entre elas.

Os limiares oficiais por célula ficaram em patamar substancialmente inferior ao observado no cenário canônico anterior. Esse comportamento é coerente com o deslocamento distribucional já documentado entre o recorte canônico e a materialização `_FULL`, especialmente no que diz respeito à participação relativa de eventos de falha. Assim, o ramo `_FULL` deve ser interpretado como um teste de robustez sob mudança de distribuição, e não como simples repetição ampliada do cenário canônico.

A célula `g` apresentou o maior número de janelas críticas e episódios, com **602 janelas críticas** e **295 episódios**. Também foi a célula com maior quantidade de janelas críticas no segmento de teste, com **291 janelas críticas**, indicando maior densidade de suporte futuro para avaliação supervisionada.

A célula `d`, por outro lado, apresentou apenas **3 janelas críticas no segmento de teste**, embora tenha 410 janelas críticas no treino. Esse é o principal ponto de atenção para as etapas posteriores, pois pode fragilizar a avaliação supervisionada da célula `d` no segmento futuro. As células `c` e `h`, com 41 e 45 janelas críticas no teste, respectivamente, também devem ser acompanhadas com atenção.

A elevada presença de episódios de uma única janela, especialmente nas células `g`, `h`, `b` e `a`, é compatível com a granularidade de 5 minutos e com limiares relativamente baixos. Esse comportamento não invalida a etapa, mas deverá ser considerado nas fases seguintes, sobretudo quando forem aplicadas regras de persistência, antecipação e construção dos rótulos supervisionados.

### Observação operacional sobre arquivamento

A execução gerou os artefatos por célula nas pastas `cell_a` a `cell_h`. O log também indica a geração dos artefatos agregados na pasta `aggregate/`, incluindo sumários por célula, thresholds, gate preliminar de modelabilidade, delta contra o cenário canônico e manifesto SHA-256.

Para fechamento reprodutível da etapa, recomenda-se arquivar também a pasta `aggregate/`, além das pastas individuais por célula, pois os artefatos agregados concentram a visão consolidada da etapa e são importantes para rastreabilidade.

### Conclusão

O **NB04_FULL está validado** para as oito células.

A etapa cumpriu seu objetivo de detectar episódios críticos por célula usando o limiar oficial `TRAIN_M2S_BY_CELL`, produzir os artefatos necessários e confirmar suporte preliminar para continuidade do pipeline `_FULL`.

O principal ponto de atenção metodológica para as próximas etapas é a distribuição desigual das janelas críticas entre treino e teste, especialmente na célula `d`, que possui suporte crítico quase totalmente concentrado no treino.

O próximo passo imediato é executar o **NB05_FULL**, preservando as validações estruturais esperadas para o núcleo temporal canônico. Em seguida, o pipeline deverá avançar para a construção e avaliação dos rótulos supervisionados, quando o gate preliminar do NB04_FULL será efetivamente convertido em decisão de modelabilidade.
